[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/cv-internship-in-a-book/blob/main/notebooks/week9_video_STARTER.ipynb)


# Week 9 -- Video — Images in Time (STARTER)
### The Computer Vision Internship | MediVision AI Lagos

**This week:** UCF-101 video — frame sampling, 2D CNN baseline, 3D convolution, temporal pooling

**Dataset:** UCF-101 video (10-class subset)

**STARTER notebook** -- your working environment. Complete each exercise before moving on.


In [ ]:
# -- Colab/Local Setup -- run this first in Colab, skip locally -------------
import os

if not os.path.exists('requirements.txt'):
    # Clone the full course repository
    !git clone https://github.com/internshipinabook/cv-internship-in-a-book.git cv-book
    %cd cv-book
    # Install all required packages (suppress verbose output with -q)
    !pip install -r requirements.txt -q

# Create working directories
os.makedirs('outputs',  exist_ok=True)   # charts, reports, predictions
os.makedirs('models',   exist_ok=True)   # trained model checkpoints
os.makedirs('datasets', exist_ok=True)   # raw downloaded data
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds --------------------------------------------------
# Setting seeds ensures every random operation produces the same result
# on every run. Essential for: comparing runs, debugging, sharing results.

import random    # Python built-in random
import numpy as np   # NumPy random (data loading, sklearn)
import torch     # PyTorch random (neural networks, dropout)

SEED = 42        # 42 is conventional -- any integer works

random.seed(SEED)           # fix Python random() calls
np.random.seed(SEED)        # fix np.random.* calls
torch.manual_seed(SEED)     # fix PyTorch CPU operations

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)  # fix GPU operations too

# Prefer deterministic cuDNN algorithms (may slow training ~5%)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Seeds fixed: {SEED} | Device: {device}')


## Learning Objectives

By the end of this week, you will be able to:

- Sample video clips into frame sequences and explain the temporal information lost at each sampling rate
- Build a 2D CNN baseline on frames (treating video as independent images)
- Build a temporal pooling model that aggregates frame-level features
- Compare 2D CNN and temporal model on UCF-101 action classification
- Write an honest analysis of what 3D convolution adds — and costs



---

## MONDAY -- "Video as Sequences — What Is Lost by Frame Sampling"


A 10-second clip at 25fps = 250 frames. Training on all 250 frames per clip is computationally prohibitive. Uniform frame sampling selects a fixed number of frames evenly spaced across the clip. At 16 frames from a 250-frame clip, one frame is sampled every 15.6 frames — roughly every 0.6 seconds. Fast actions (a punch, a frame in tennis) may fall between samples entirely.


### Exercise 9.1 -- Frame sampling rate analysis

For 5 different UCF-101 action classes, compute the average clip length in seconds. For each, calculate: at 8-frame sampling, what is the minimum detectable action duration? Which actions are likely to be missed?


In [ ]:
# Exercise 9.1: Frame sampling rate analysis
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Monday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Monday: Video as Sequences — What Is Lost by Frame Sampling (scaffold) --
import cv2
import torch

def get_clip_info(path):
    cap = cv2.VideoCapture(path)
    fps    = cap.get(cv2.CAP_PROP_FPS)
    frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    dur    = frames / fps
    cap.release()
    return {"fps": fps, "frames": int(frames), "duration_s": dur}

# Examine a sample clip
info = get_clip_info("datasets/ucf101/Basketball/v_Basketball_g01_c01.avi")
print(f"FPS: {info['fps']}  |  Frames: {info['frames']}  |  Duration: {info['duration_s']:.1f}s")
print(f"At 16-frame sampling, interval = {info['frames']/16:.0f} frames = {info['duration_s']/16:.2f}s")


### Monday Morning Moment

*Slack — Monday, 11:00am.*

**Ngozi Eze-Okafor:** At 16-frame sampling on a 250-frame clip, I'm sampling every 0.6 seconds. A basketball shot takes about 0.3 seconds.

**Dr. Kwame Asante:** So you might miss the moment of release entirely.

**Ngozi Eze-Okafor:** Yes. The model would see the player running, then the follow-through, without the release.

**Yewande Adeyemi:** That's why you need more frames for fast actions. 32 or 64.

**Dr. Kwame Asante:** And what does doubling the frame count cost?

**Yewande Adeyemi:** Twice the memory. Twice the compute. Twice the training time.

**Dr. Kwame Asante:** Every architecture decision is a tradeoff. Write the tradeoff in the notebook before you choose.




---

## TUESDAY -- "2D CNN Baseline — Frames as Independent Images"


The simplest video model: extract N frames, run a 2D CNN on each independently, average the predictions. This completely ignores temporal order — "picking up a ball" and "throwing a ball" produce similar single-frame distributions but opposite temporal sequences.


### Exercise 9.2 -- 2D CNN training

Train VideoFrameCNN on the UCF-101 10-class subset for 10 epochs. Report top-1 accuracy, top-3 accuracy, training time per epoch. Which 3 classes have the lowest per-class accuracy?


In [ ]:
# Exercise 9.2: 2D CNN training
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Tuesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Tuesday: 2D CNN Baseline — Frames as Independent Images (scaffold) --
class VideoFrameCNN(nn.Module):
    """2D CNN baseline for video classification.
    Treats each frame independently; averages predictions across frames.
    Input: (B, T, C, H, W) — batch, time, channel, height, width
    Output: (B, num_classes)
    """
    def __init__(self, num_classes=10, n_frames=16):
        super().__init__()
        backbone = models.resnet18(weights="IMAGENET1K_V1")
        self.features   = nn.Sequential(*list(backbone.children())[:-1])
        self.classifier = nn.Linear(512, num_classes)
    
    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B*T, C, H, W)          # merge batch and time
        feats = self.features(x).flatten(1)  # (B*T, 512)
        feats = feats.view(B, T, -1).mean(1) # average across time
        return self.classifier(feats)



---

## WEDNESDAY -- "Temporal Pooling — Keeping Time in the Model"


Instead of averaging predictions, aggregate frame features before classification. This allows the model to weight some frames more than others based on their content. The simplest version: max-pool across the time dimension (keep the most activated feature per position). More expressive: learn a temporal attention weight per frame.


### Exercise 9.3 -- Temporal model comparison

Train TemporalPoolingCNN with the same hyperparameters. Compare to 2D baseline. For the 3 weakest classes from 9.2: does temporal pooling help any of them? Why or why not?


In [ ]:
# Exercise 9.3: Temporal model comparison
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Wednesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Wednesday: Temporal Pooling — Keeping Time in the Model (scaffold) --
class TemporalPoolingCNN(nn.Module):
    """Temporal pooling over frame features.
    Max-pools across time dimension to keep strongest activations.
    """
    def __init__(self, num_classes=10):
        super().__init__()
        backbone = models.resnet18(weights="IMAGENET1K_V1")
        self.features   = nn.Sequential(*list(backbone.children())[:-1])
        self.classifier = nn.Linear(512, num_classes)
    
    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B*T, C, H, W)
        feats = self.features(x).flatten(1).view(B, T, -1)
        # Max-pool across time: keep strongest feature per position
        pooled, _ = feats.max(dim=1)  # (B, 512)
        return self.classifier(pooled)



---

## THURSDAY -- "3D Convolution — What It Adds and Costs"


A 3D convolutional filter has shape (C_out, C_in, D, H, W) where D is the temporal depth. A 3×3×3 filter sees 3 consecutive frames simultaneously. This captures local motion patterns directly in the filter weights. The cost: memory and compute scale with D. A 16-frame input to a 3D ResNet requires roughly 8× the memory of the same input to a 2D ResNet. I3D (Inflated 3D ConvNet, Carreira & Zisserman 2017) and SlowFast (Feichtenhofer et al. 2019) are the standard architectures.


### Exercise 9.4 -- 3D convolution parameter count

Compute: how many parameters does a single 3D Conv layer (C_in=64, C_out=128, kernel=3×3×3) have? Compare to the 2D equivalent (C_in=64, C_out=128, kernel=3×3). At this ratio, estimate total parameters for a 3D ResNet-18. Is this feasible on a 4GB GPU?


In [ ]:
# Exercise 9.4: 3D convolution parameter count
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Thursday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Thursday: 3D Convolution — What It Adds and Costs (scaffold) --
# 3D convolution — understand the shape, do not necessarily train
import torch.nn as nn

conv3d = nn.Conv3d(in_channels=3, out_channels=64,
                   kernel_size=(3,3,3), padding=(1,1,1))

# Input: (B=4, C=3, T=16, H=112, W=112)
x = torch.randn(4, 3, 16, 112, 112)
out = conv3d(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")  # (4, 64, 16, 112, 112)
print(f"Parameters in this one layer: {conv3d.weight.numel():,}")
# Compare: 2D equivalent: nn.Conv2d(3,64,3) → 1,728 params
# 3D: (3,3,3) → 1,728 × 3 = 5,184 params per filter



---

## FRIDAY -- "The Friday Build: Video Model Comparison"


Train and evaluate: (1) 2D CNN frame averaging, (2) temporal max-pooling CNN, on the 10-class UCF-101 subset. Report top-1 accuracy and top-3 accuracy for each. Write a one-paragraph analysis: where does the temporal model help, and where does the 2D baseline fail? Name two action classes where temporal order is essential.


**Friday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Friday: The Friday Build: Video Model Comparison (scaffold) --
# Video model comparison report:
# 1. Top-1 and top-3 accuracy for each model
# 2. Per-class breakdown: which actions benefit most from temporal pooling?
# 3. Two actions where temporal order is essential (2D baseline fails)
# 4. Estimated training time for a full 3D CNN on your hardware
# 5. Recommendation: when is temporal modelling worth the compute cost?


### Friday Workplace Moment

*MediVision AI — Friday standup.*

**Ngozi Eze-Okafor:** 2D CNN frame averaging: 72.3% top-1. Temporal max-pooling: 76.8% top-1. Gain: 4.5 percentage points.

**Dr. Kwame Asante:** Which classes improved most?

**Ngozi Eze-Okafor:** "Archery" and "Basketball" — both require seeing the full motion sequence, not just one frame. "Band marching" improved least — a single frame is almost as informative as the sequence.

**Yewande Adeyemi:** Archery makes sense. You cannot tell from one frame whether the arrow has been released or is still being drawn.

**Dr. Kwame Asante:** Clinical corollary: which surgical actions would require temporal context to classify correctly?

**Ngozi Eze-Okafor:** Any action with pre- and post-states: cutting vs holding, suturing vs preparing to suture.

**Dr. Kwame Asante:** That is a research question. Write it in the notebook.



## YOUR WEEK 9 CHECKLIST

Check each item before moving to the next week.
If you cannot explain an item without notes, revisit that section.

- [ ] I can compute how many frames per second are sampled at a given rate and identify which actions may be missed.
- [ ] I trained both the 2D baseline and temporal pooling model and can state the top-1 accuracy difference from memory.
- [ ] I know what a 3D convolutional filter does differently from a 2D filter — specifically, what temporal information it captures.
- [ ] I named two action classes where temporal order is essential for classification.
- [ ] I can estimate 3D CNN parameter count and assess feasibility on a given GPU.


## Challenge Task

> **Core path:** Implement a learnable temporal attention mechanism: instead of max-pooling, learn a scalar attention weight per frame. Does this outperform max-pooling? Visualise the learned attention weights for 5 clips.

> **Advanced path:** Implement optical flow as an additional input channel alongside RGB frames. Use OpenCV's calcOpticalFlowFarneback(). Does adding optical flow improve the 2D CNN baseline?


## Common Mistakes

**Treating video classification as independent frame classification:** Averaging frame predictions loses temporal order. "Person about to punch" and "person finishing a punch" produce similar single-frame features but should produce different predictions.

**Using 3D convolution without sufficient memory:** A single 16-frame, 112×112 input to a 3D ResNet requires ~6GB GPU memory. Check hardware constraints before choosing the architecture.

**Sampling frames without checking action duration:** Actions shorter than the sampling interval cannot be detected. Always compute the minimum detectable action duration for your sampling rate.



---

## Get the Full Book

This notebook is part of **The Computer Vision Internship** -- Book 3 of 9, InternshipInABook(tm).

Covers: natural images, medical X-rays, breast & uterine cancer, video, GradCAM/LIME/SHAP/Integrated Gradients, and fairness audits.

Available at [internshipinabook.com](https://internshipinabook.com)
